In [2]:
import pandas as pd

url = "https://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
df = pd.read_csv(url, sep=';', encoding='latin1', low_memory=False)

mask = (
    df['DENOM_SOCIAL'].str.contains('MASTER', na=False, case=False) |
    df['GESTOR'].str.contains('MASTER', na=False, case=False) |
    df['ADMIN'].str.contains('MASTER', na=False, case=False)
)

fundos = df[mask][['CNPJ_FUNDO', 'DENOM_SOCIAL', 'TP_FUNDO',
                   'GESTOR', 'ADMIN', 'SIT', 'VL_PATRIM_LIQ']].copy()

print(f"Fundos relacionados ao Master: {len(fundos)}")
print(fundos.to_string(index=False))
fundos.to_csv("data/fundos_master.csv", index=False)

Fundos relacionados ao Master: 1239
        CNPJ_FUNDO                                                                                         DENOM_SOCIAL TP_FUNDO                                                                               GESTOR                                                                    ADMIN                     SIT  VL_PATRIM_LIQ
00.016.959/0001-15                                         FUNDO DE APLIC EM QUOTAS DE FUNDOS DE INV BMC 60 DIAS MASTER   FACFIF                                                                                  NaN                                                                      NaN               CANCELADA            NaN
00.290.170/0001-58                                          FDO DE APLIC EM QUOTAS DE FDOS DE INV HSBC MASTER DI DIARIO   FACFIF                                                                                  NaN                                                                      NaN               CANCELADA  

In [3]:
import pandas as pd

url = "https://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv"
df = pd.read_csv(url, sep=';', encoding='latin1', low_memory=False)

cnpjs_master = [
    "33923798000100",  # Banco Master S.A.
    "33884941000194",  # Banco Master Múltiplo
    "09526594000143",  # Banco Master de Investimento
]

def limpar_cnpj(col):
    return col.astype(str).str.replace(r'\D', '', regex=True)

mask = (
    limpar_cnpj(df['CNPJ_ADMIN']).isin(cnpjs_master) |
    limpar_cnpj(df['CPF_CNPJ_GESTOR']).isin(cnpjs_master) |
    limpar_cnpj(df['CNPJ_CUSTODIANTE']).isin(cnpjs_master) |
    limpar_cnpj(df['CNPJ_CONTROLADOR']).isin(cnpjs_master) |
    df['CUSTODIANTE'].str.contains('MASTER', na=False, case=False) |
    df['ADMIN'].str.contains('BANCO MASTER', na=False, case=False)
)

fundos = df[mask][['CNPJ_FUNDO', 'DENOM_SOCIAL', 'TP_FUNDO',
                   'GESTOR', 'ADMIN', 'CUSTODIANTE', 'SIT', 'VL_PATRIM_LIQ']].copy()

fundos = fundos[fundos['SIT'] != 'CANCELADA']  # só ativos ou em liquidação

print(f"Fundos ativos/liquidação com vínculo Master: {len(fundos)}")
print(fundos.to_string(index=False))
fundos.to_csv("data/fundos_master_filtrado.csv", index=False)

Fundos ativos/liquidação com vínculo Master: 7
        CNPJ_FUNDO                                                                       DENOM_SOCIAL TP_FUNDO                                         GESTOR                                                         ADMIN                                                   CUSTODIANTE                     SIT  VL_PATRIM_LIQ
41.709.776/0001-62 MAM SELEÇÃO FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTOS MULTIMERCADO       FI MAM ASSET MANAGEMENT GESTORA DE RECURSOS LTDA. MASTER S/A CORRETORA DE CAMBIO, TITULOS E VALORES MOBILIARIOS MASTER S/A CORRETORA DE CAMBIO, TITULOS E VALORES MOBILIARIOS    FASE PRÉ-OPERACIONAL            NaN
02.727.085/0001-30                                                      MÁXIMA  FMP - FGTS  PETROBRAS       FI                 ACURA GESTORA DE RECURSOS LTDA MASTER S/A CORRETORA DE CAMBIO, TITULOS E VALORES MOBILIARIOS MASTER S/A CORRETORA DE CAMBIO, TITULOS E VALORES MOBILIARIOS EM FUNCIONAMENTO NORMAL     1

In [6]:
import requests, json

r = requests.get("https://minhareceita.org/16685929000131", timeout=10)
dados = r.json()

# QSA
print("=== SÓCIOS/ADMINISTRADORES ===")
for socio in dados.get('qsa', []):
    print(f"  {socio['nome_socio']}")
    print(f"  Qualificação: {socio['qualificacao_socio']}")
    print(f"  Identificador: {socio['identificador_de_socio']} (1=PJ, 2=PF)")
    print(f"  CNPJ/CPF: {socio['cnpj_cpf_do_socio']}")
    print(f"  Entrada: {socio['data_entrada_sociedade']}")
    print()

# dados gerais
print(f"Capital social: R$ {dados.get('capital_social'):,.2f}")
print(f"Município: {dados.get('municipio')} / {dados.get('uf')}")
print(f"CNAE: {dados.get('cnae_fiscal_descricao')}")

=== SÓCIOS/ADMINISTRADORES ===
Capital social: R$ 0.00
Município: RIO DE JANEIRO / RJ
CNAE: Fundos de investimento, exceto previdenciários e imobiliários


In [7]:
import requests, zipfile, io, pandas as pd

url = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_2025.zip"
r = requests.get(url, timeout=60)

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    print(z.namelist())

['inf_mensal_fii_ativo_passivo_2025.csv', 'inf_mensal_fii_complemento_2025.csv', 'inf_mensal_fii_geral_2025.csv']


In [9]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import webbrowser, os

# ── 1. Carrega o cadastro de FIIs ────────────────────────────────────────────
url = "https://dados.cvm.gov.br/dados/FII/CAD/DADOS/cad_fii.csv"
df_fii = pd.read_csv(url, sep=';', encoding='latin1', low_memory=False)

# normaliza CNPJs
def limpar_cnpj(s):
    return str(s).replace('.','').replace('/','').replace('-','').strip()

df_fii['CNPJ_FUNDO_LIMPO'] = df_fii['CNPJ_FUNDO'].apply(limpar_cnpj)
df_fii['CNPJ_ADMIN_LIMPO'] = df_fii['CNPJ_ADMINISTRADOR'].apply(limpar_cnpj)

# ── 2. Filtra fundos ligados ao Master ───────────────────────────────────────
cnpjs_master = {
    "33923798000100": "Banco Master S/A",
    "33884941000194": "Banco Master Múltiplo",
    "09526594000143": "Banco Master de Investimento",
    "33886862000112": "Master S/A Corretora",
    "58497702000102": "Banco Letsbank",
}

mask = df_fii['CNPJ_ADMIN_LIMPO'].isin(cnpjs_master.keys())
fundos = df_fii[mask].copy()

print(f"Fundos administrados pelo grupo Master: {len(fundos)}")
print(fundos[['CNPJ_FUNDO', 'DENOM_SOCIAL', 'NM_ADMINISTRADOR', 'NM_GESTOR', 'SIT']].to_string())

HTTPError: HTTP Error 404: Not Found

In [8]:
import requests, zipfile, io
import pandas as pd

url = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_2025.zip"
r = requests.get(url, timeout=60)

cnpj_macam = "16.685.929/0001-31"

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for nome in z.namelist():
        with z.open(nome) as f:
            df = pd.read_csv(f, sep=';', encoding='latin1', low_memory=False)

            # descobre qual coluna tem o CNPJ
            col_cnpj = [c for c in df.columns if 'CNPJ' in c.upper()][0]

            filtrado = df[df[col_cnpj] == cnpj_macam]

            print(f"\n{'='*60}")
            print(f"Arquivo: {nome}")
            print(f"Registros encontrados: {len(filtrado)}")

            if len(filtrado) > 0:
                print(filtrado.T.to_string())  # transpõe pra ver todos os campos


Arquivo: inf_mensal_fii_ativo_passivo_2025.csv
Registros encontrados: 12
                                                         1588                1589                1590                1591                1592                1593                1594                1595                1596                1597                1598                1599
CNPJ_Fundo_Classe                          16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31  16.685.929/0001-31
Data_Referencia                                    2025-01-01          2025-02-01          2025-03-01          2025-04-01          2025-05-01          2025-06-01          2025-07-01          2025-08-01          2025-09-01          2025-10-01          2025-11-01          2025-12-01
Versao                                                      3                   